<a href="https://colab.research.google.com/github/rcsb/rcsb-training-resources/blob/master/training-events/2026/python-rcsb-api/advanced_search_and_data_api.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install `rcsb-api`
%pip install --upgrade rcsb-api

# Install `rich` for pretty printing
%pip install rich

# Advanced Search and Data API Usage

Building upon existing documentation and tutorials for utilizing the Search and Data API modules of the `rcsb-api` package, this notebook is intended to provide some additional, more advanced use cases that will help you take full advantage of all the tooling and options available.

Specifically, this tutorial will cover:
1. Performing structure-similarity searches with the [new AI-powered 3D search service](https://www.rcsb.org/news/feature/69bc5e328557ae0f261bc71c) at RCSB.org
2. Grouping of search results based on sequence similarity
3. Fine-tuning of `rcsb-api` package configurations for optimal performance

## 1. Performing structure-similarity searches with the new AI-powered 3D search service
RCSB.org recently introduced a [new AI-powered 3D search service](https://www.rcsb.org/news/feature/69bc5e328557ae0f261bc71c) service, which can perform archive wide similarity searches at efficient speed.

Along with this release also introduced several changes in the parameters provided to the [`StructSimilarityQuery`](https://rcsbapi.readthedocs.io/en/latest/search_api/query_construction.html#structure-similarity-search). Specifically, the updated version of the class adds the following parameters: `similarity_type`, `number_of_candidates`, and `ptmscore_cutoff`. For details on all available parameters, see the [documentation](https://rcsbapi.readthedocs.io/en/latest/search_api/query_construction.html#structure-similarity-search).

Below is an example of searching for structures of similar 3D similarity to PDB ID 4HHB:

In [ ]:
from rcsbapi.search import StructSimilarityQuery

q1 = StructSimilarityQuery(
    structure_search_type="entry_id",
    entry_id="4HHB",
    assembly_id="1",
    target_search_space="assembly",
    similarity_type="local",  # Search mode ("local" or "global"); "local" favors matches to specific regions of a structure, "global" applies length normalization to favor global similarity
    number_of_candidates=10000,  # Controls the max number of similar matches to return (0 to 15000)
    ptmscore_cutoff=0.8,  # Minimum predicted TM-score threshold, above which hits will be returned (0.0 to 1.0)
)

# Return assembly IDs
results = list(q1("assembly"))
print(f"Number of search results: {len(results)}")
print(f"First 10 similar assemblies: {results[:10]}")

You can also provide your own local structure files for performing archive-wide searches, either by providing the local file path or a URL:

In [ ]:
from rcsbapi.search import StructSimilarityQuery

# Using `file_path` - uncomment once a real input file path is provided in query below
# q2 = StructSimilarityQuery(
#     structure_search_type="file_upload",
#     file_path="/PATH/TO/FILE.cif",  # specify local model file path
#     file_format="cif"
# )
# results = list(q2())
# print(f"Number of search results: {len(results)}")

# Using file_url
q3 = StructSimilarityQuery(
    structure_search_type="file_url",
    file_url="https://files.rcsb.org/download/4HHB.cif",
    file_format="cif",
    assembly_id="1",
    target_search_space="assembly",
    similarity_type="local",
    number_of_candidates=10000,
    ptmscore_cutoff=0.8,
)
results = list(q3("assembly"))
print(f"Number of search results: {len(results)}")
print(f"First 10 similar assemblies: {results[:10]}")

## 2. Grouping of search results by sequence identity
Search API results can be [grouped by various strategies](https://rcsbapi.readthedocs.io/en/latest/search_api/additional_examples.html#groupby-example), such as by sequence identity or UniProt accession code, in order to consolidate the total number of returned results and obtain a "representative" structure of highly-similar entities.

To demonstrate this, let's assume the following use-case, in which we want to:
1) Use the Search API to group all protein entities in PDB from *Arabidopsis thaliana* together by sequence identity of 90%, and return a list of just the representative structure of each group
2) Use the Data API to retrieve the associated 1-letter sequence and cluster ID of all represntatives


For step (1), we perform a Search API query using the `group_by` parameter (with a `GroupBy` object), as follows:

In [ ]:
from rcsbapi.search import search_attributes as attrs
from rcsbapi.search import GroupBy

# Query all PDB entries of type "Protein" and from Arabidopsis thaliana
qA = attrs.entity_poly.rcsb_entity_polymer_type == "Protein"
qB = attrs.rcsb_entity_source_organism.scientific_name == "Arabidopsis thaliana"
query = qA & qB

# First print the total number of polymer entities *without* grouping
ungrouped_search_results = list(query(return_type="polymer_entity"))
print(f"Total number of polymer entities (ungrouped): {len(ungrouped_search_results)}")

# Now perform the same query but with grouping
search_results = list(
    query(
        group_by=GroupBy(
            aggregation_method="sequence_identity",
            similarity_cutoff=90,
        ),
        group_by_return_type="representatives",
        return_type="polymer_entity",
    )
)
print(f"Total number of group representatives: {len(search_results)}")
print(f"First 5 representatives: {search_results[0:5]}")  # will contain representative entities like ['104M_1', '10AF_1', '10BM_1', '10FT_2', '10GH_3']

The "representatives" returned above will just be one member of the group, with no particular ranking applied. If you wish to apply a particular ranking (e.g., to pick the best resolution structure of each group), you can do so using the `RankingCriteriaType` and specifying which attribute to use for ranking/sorting (see [usage example here](https://rcsbapi.readthedocs.io/en/latest/search_api/additional_examples.html#sequence-identity)). Please note, however, that doing so may slow down the query if the result set is very large.

Next, for step (2), the data retrieval can be done as follows:

In [ ]:
from rcsbapi.data import DataQuery as Query
from rich import print as rprint

# Constructing the Query object
query = Query(
    input_ids=search_results,  # provide `search_results` from Search API query in step (1) as input to Data API query
    input_type="polymer_entities",
    return_data_list=[
        # Both `cluster_id` and `identity` are needed to determine the correct cluster ID
        "rcsb_cluster_membership.cluster_id",
        "rcsb_cluster_membership.identity",
        #
        "entity_poly.pdbx_seq_one_letter_code_can",  # 1-letter sequence
    ]
)

# Executing the query
data_results = query.exec()
print("Total number of data results: ", len(data_results["data"]["polymer_entities"]))
rprint("First result: ", data_results["data"]["polymer_entities"][0])

This will return the requested set of data for the representatives. Note that the returned set of `rcsb_cluster_membership` data will contain the `cluster_id`s for all `identity` levels, so it will be necessary to post-process the data to extract out the correct `cluster_id` for the `identity` of `90`.

## 3. Fine-tuning of `rcsb-api` package configurations for optimal performance
A variety of [configuration options exist](https://rcsbapi.readthedocs.io/en/latest/config/custom_configuration.html) for the `rcsb-api` package that you can adjust to control its performance, such as the number of requests per second, the API timeout, as well as the input ID limit for certain modules.

These are especially useful for fine-tuning your scripts to improve the throughput (or, vice-versa, reduce the aggressiveness in the event of timeouts or other issues).

For example, to increase the number of concurrent Data API requests to run, you can configure this:

In [ ]:
from rcsbapi.config import config

# Override the default number of concurrent Data API requests
config.DATA_API_MAX_CONCURRENT_REQUESTS = 8

Now, let's compare how this impacts the overall time to run a Data API query on 4,500 structures:

In [ ]:
# Measure total query time with concurrency of 8
from rcsbapi.data import DataQuery as Query
from rcsbapi.config import config
import time

# Constructing the Query object
data_query = Query(
    input_ids=ungrouped_search_results,  # input of ~4,500 entities (from ungrouped example above)
    input_type="polymer_entities",
    return_data_list=[
        "rcsb_cluster_membership.cluster_id",
        "rcsb_cluster_membership.identity",
        "entity_poly.pdbx_seq_one_letter_code_can",
    ]
)

# Set concurrency to 8 (resetting here in case cells are run out of order)
config.DATA_API_MAX_CONCURRENT_REQUESTS = 8

# Executing the query
start_time = time.time()
data_results = data_query.exec(progress_bar=True)
print(f"\nQuery duration: {time.time() - start_time:.3f} seconds")
print("Total number of data results: ", len(data_results["data"]["polymer_entities"]))

...versus a concurrency of 2:

In [ ]:
# Set concurrency of 2
config.DATA_API_MAX_CONCURRENT_REQUESTS = 2

# Executing the query
start_time = time.time()
data_results = data_query.exec(progress_bar=True)
print(f"\nQuery duration: {time.time() - start_time:.3f} seconds")
print("Total number of data results: ", len(data_results["data"]["polymer_entities"]))

#### ***Important:*** Please adjust these settings with care and caution! Attempting to increase them too much may end up hindering performance and/or exceeding RCSB.org API rate limits or server timeouts.